# Classifier evaluation queries

Issue: #11

## Question

The query classifier (#34) needs text queries with known true categories, and the ranking queries (#10) cannot supply them because OTTO has no query text. What set of search terms can we test it on, including hard cases?

## Data used

Inputs: `artifacts/classifier_queries_v1_raw.json` (the authored seed terms), `artifacts/category_names.json` (#9).

Outputs: `artifacts/classifier_queries_v1.jsonl` and `artifacts/classifier_queries_v1.meta.json`. Kept **separate** from the ranking queries of #10.

**Where the terms came from.** No API key was used. The seed terms were written directly by Claude (claude-sonnet-5) in the Claude Code session on 2026-09-19 and saved once as the raw file, like the taxonomy of #9. They are fictional search terms for the synthetic categories and are not derived from OTTO. This notebook only reads the raw file and applies deterministic rules, so it never calls an LLM and cannot change between runs.

**Kinds of query.**
- `paraphrase`: for each category, 6 hand-written synonyms that do not contain the category name, and 12 templated variants (two per synonym, from `buy X`, `X online`, `best X`, `cheap X`, `X sale`, `X uk`).
- `misspelling`: 6 per category, made by a seeded rule that deletes, doubles, swaps or replaces one letter of a synonym.
- `multi-intent`: 3 per category, two synonyms from two categories in different groups joined by `and`. **Two** true categories.
- `out_of_scope`: 101 hand-written terms for things the shop does not sell (electronics, food, cars, pets, tools, services...), with templated and misspelled variants. True categories: none.

**Splits.** Every query goes to `dev` (about 20 percent, for prompt tuning) or `test` (about 80 percent) by a hash of its text. Never tune on the test part.

In [ ]:
import hashlib
import json
import random
from collections import Counter
from pathlib import Path

import polars as pl

ART = Path("artifacts") if Path("artifacts").exists() else Path("../artifacts")
raw = json.loads((ART / "classifier_queries_v1_raw.json").read_text())
names = json.loads((ART / "category_names.json").read_text())["categories"]
name_to_id = {v["name"]: int(k) for k, v in names.items()}
group_of = {v["name"]: v["group"] for v in names.values()}
syn = raw["paraphrases_by_category"]
assert set(syn) == set(name_to_id), "the raw file must cover exactly the 80 categories"
TEMPLATES = raw["templates"]
print(len(syn), "categories,", sum(len(v) for v in syn.values()), "hand-written synonyms,", len(raw["out_of_scope_terms"]), "out-of-scope terms")

## Build the queries with deterministic rules

In [ ]:
def stable_int(text, mod):
    return int(hashlib.md5(text.encode()).hexdigest(), 16) % mod


def misspell(text, key):
    """One seeded typo in a word of at least 4 letters: delete, double, swap or replace a letter."""
    rng = random.Random(stable_int(key + text, 2**31))
    words = text.split()
    candidates = [i for i, w in enumerate(words) if len(w) >= 4 and w.isalpha()]
    if not candidates:
        return None
    i = rng.choice(candidates)
    w = words[i]
    p = rng.randrange(1, len(w) - 1)
    op = rng.choice(["delete", "double", "swap", "replace"])
    if op == "delete":
        w2 = w[:p] + w[p + 1:]
    elif op == "double":
        w2 = w[:p] + w[p] + w[p:]
    elif op == "swap":
        w2 = w[:p] + w[p + 1] + w[p] + w[p + 2:]
    else:
        w2 = w[:p] + rng.choice("aeiou".replace(w[p], "")) + w[p + 1:]
    if w2 == w:
        return None
    words[i] = w2
    return " ".join(words)


rows = []
def add(query, cats, oos, kind):
    rows.append({"query": query, "true_categories": cats, "out_of_scope": oos, "kind": kind})


for cat, terms in syn.items():
    for t in terms:
        add(t, [cat], False, "paraphrase")
    for i, t in enumerate(terms):                     # two templated variants per synonym
        add(TEMPLATES[i].format(s=t), [cat], False, "paraphrase")
        add(TEMPLATES[(i + 3) % 6].format(s=t), [cat], False, "paraphrase")
    for t in terms:
        m = misspell(t, "cat")
        if m:
            add(m, [cat], False, "misspelling")

rng = random.Random(0)
cats_sorted = sorted(syn)
for cat in cats_sorted:
    others = [c for c in cats_sorted if group_of[c] != group_of[cat]]
    for k, partner in enumerate(rng.sample(others, 3)):
        add(f"{syn[cat][k % 6]} and {syn[partner][(k + 2) % 6]}", [cat, partner], False, "multi-intent")

for term in raw["out_of_scope_terms"]:
    add(term, [], True, "out_of_scope")
    for tpl in ("buy {s}", "{s} online", "cheap {s}"):
        add(tpl.format(s=term), [], True, "out_of_scope")
oos_rows = [r for r in rows if r["out_of_scope"]]
for r in random.Random(1).sample(oos_rows, 60):
    m = misspell(r["query"], "oos")
    if m:
        add(m, [], True, "out_of_scope")

# deduplicate case-insensitively, keep the first occurrence
seen, unique = set(), []
for r in rows:
    key = r["query"].lower().strip()
    if key not in seen:
        seen.add(key)
        unique.append(r)
print(f"{len(rows)} generated, {len(unique)} after removing duplicates")

for i, r in enumerate(unique):
    r["id"] = i
    r["true_cluster_ids"] = [name_to_id[c] for c in r["true_categories"]]
    r["split"] = "dev" if stable_int(r["query"].lower(), 5) == 0 else "test"
df = pl.DataFrame(unique).select("id", "query", "true_categories", "true_cluster_ids", "out_of_scope", "kind", "split")

## Checks

In [ ]:
cat_names_lower = {c.lower() for c in syn}
q_lower = df["query"].str.to_lowercase()
assert q_lower.n_unique() == df.height, "duplicate queries (case-insensitive)"
assert not (set(q_lower.to_list()) & cat_names_lower), "a query equals a category name"
assert df.filter(pl.col("kind") == "multi-intent")["true_categories"].list.len().min() == 2
assert (df.filter(pl.col("out_of_scope"))["true_categories"].list.len() == 0).all()
assert (df.filter(~pl.col("out_of_scope"))["true_categories"].list.len() >= 1).all()

test = df.filter(pl.col("split") == "test")
per_cat = Counter(c for cats in test.filter(~pl.col("out_of_scope"))["true_categories"].to_list() for c in cats)
assert len(per_cat) == 80 and min(per_cat.values()) >= 15, min(per_cat.values())
n_oos_test = test.filter(pl.col("out_of_scope")).height
assert n_oos_test >= 150, n_oos_test
print(f"every category has at least {min(per_cat.values())} test queries (median {sorted(per_cat.values())[40]}); {n_oos_test} out-of-scope test queries")
print(df.group_by("kind", "split").len().sort("kind", "split").pivot(on="split", index="kind", values="len"))
print(df.group_by("split").len())

## Spot check of the out-of-scope queries

The out-of-scope terms were written to fit no category, but one may still describe something the shop sells. 30 out-of-scope test queries are drawn with a fixed seed and judged by hand: does the query describe something that belongs to one of the 80 categories?

**Review history.** First pass (before the change below): 1 of 30 sampled queries described an in-scope product (`cheap board shorts surf` is swimwear). All out-of-scope terms were then reviewed and 5 ambiguous ones were removed (`board shorts surf`, `football boots studded`, `swimming goggles pool`, `light bulbs`, `nappies`) and 5 clear ones added. The check below is the **second pass on a fresh sample**.

In [ ]:
sample = test.filter(pl.col("out_of_scope")).sample(30, seed=0, shuffle=True)
for i, q in enumerate(sample["query"].to_list()):
    print(f"{i + 1:>2}. {q}")

In [ ]:
# Judged by hand (Claude, in this session) after reading the 30 queries printed above.
# Query 1, `cheap monitor stand`, is borderline: a monitor stand could be filed under Desks. The other 29 clearly fit no category.
SPOT_CHECK_FITS_A_CATEGORY = [1]          # numbers of the listed queries that describe an in-scope product
error_rate = len(SPOT_CHECK_FITS_A_CATEGORY) / 30
print(f"spot check: {len(SPOT_CHECK_FITS_A_CATEGORY)} of 30 out-of-scope queries fit a category (error rate {error_rate:.0%})")

In [ ]:
out = ART / "classifier_queries_v1.jsonl"
out.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in df.to_dicts()) + "\n")
meta = {
    "version": "v1", "date": "2026-09-19", "author_model": "claude-sonnet-5 (written in the Claude Code session, no API)",
    "source_seed_file": "artifacts/classifier_queries_v1_raw.json", "queries": df.height,
    "by_split": dict(df.group_by("split").len().iter_rows()), "by_kind": dict(df.group_by("kind").len().iter_rows()),
    "min_test_queries_per_category": min(per_cat.values()), "out_of_scope_test_queries": n_oos_test,
    "spot_check": {"n": 30, "fit_a_category": SPOT_CHECK_FITS_A_CATEGORY, "error_rate": error_rate},
    "note": "Fictional terms for the synthetic categories of issue #9. Tune prompts on dev only; report on test only.",
}
(ART / "classifier_queries_v1.meta.json").write_text(json.dumps(meta, indent=2))
lines = [json.loads(ln) for ln in out.read_text().splitlines()]
assert len(lines) == df.height and {"query", "true_categories", "out_of_scope", "kind", "split"} <= set(lines[0])
print("saved", out.name, "with", len(lines), "lines; each parses as JSON")

## Known limits

- The terms are written by one author and the misspellings come from a simple rule, so they are easier than real typos. Multi-word paraphrases were written not to contain the category name, but some categories are close (for example `Trousers & Jeans` and `Chinos & Jeans`, or `Trainers` and `Kids Shoes`), and a few queries may fit two categories. Multi-intent queries are the only ones with two labels.
- **About 3 percent of the out-of-scope queries may be borderline.** The second spot check found 1 of 30 (`monitor stand`, which could be Desks). The list was not edited again, so the sampled hit remains in the data. Treat out-of-scope precision and recall on this set as accurate to a few percent at best.
- All category names are synthetic (#9), so accuracy on this set measures the classifier as a pipeline component, not real query understanding.

## What I learned

To be written by the owner of the project.